# Does cropping explain the detectors' scores? — full AI-GenBench set

**The question in plain words:** every image is scored on a 200×200 pixel crop, never the full
picture. For a small image, that is almost the *whole* image; for a big photo, a tiny slice.
Real photos here tend to be larger than fakes, so "how much of the picture survives the crop"
differs between the classes. If a detector could key off *that* instead of content, its scores
would be worthless as evidence.

**Four steps:**
1. Could crop fraction **by itself**, with no pixels at all, tell real from fake? (Is the
   shortcut *available*?)
2. Do the detectors' scores **move with it** among real photos?
3. **Remove the shortcut** (compare only images cropped by a similar amount) and see whether the
   detectors still separate real from fake.
4. Look at *which* real photos cause trouble.

No detector is re-run — this reads the 355,638-image score table and the crop geometry recorded
in `manifest.csv`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()

DETECTORS = ["cnndetection", "univfd",
             "dmimagedetection_progan", "dmimagedetection_latent",
             "aeroblade"]

REAL, FAKE = "#3b6ea5", "#c4462c"
plt.rcParams.update({
    "figure.dpi": 120, "font.size": 9, "axes.grid": True,
    "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 9, "axes.titlelocation": "left", "axes.titleweight": "bold",
})

def auroc(score, label):
    '''Rank-based AUROC; no sklearn in this env. score first, label second -- same
    convention as merge_scores.ipynb.'''
    r = pd.Series(score).rank()
    npos, nneg = (label == 1).sum(), (label == 0).sum()
    if npos == 0 or nneg == 0:
        return np.nan
    return (r[label == 1].sum() - npos * (npos + 1) / 2) / (npos * nneg)

def spearman(a, b):
    '''Rank correlation; no scipy in this env.'''
    a, b = pd.Series(a).rank(), pd.Series(b).rank()
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))

# The geometry columns travel in master_scores.csv itself when the manifest carried them
# (build_manifest.py writes them since ec1b4f6), and are joined from the manifest only when
# it did not. Merging unconditionally suffixes both copies to _x/_y and every later
# M.source_width raises AttributeError -- which is how this first failed on the full run.
M = pd.read_csv(ROOT / "scores" / "master_scores.csv", low_memory=False)
_need = [c for c in ("source_width", "source_height") if c not in M.columns]
if _need:
    M = M.merge(pd.read_csv(ROOT / "manifest.csv", low_memory=False)[["image_id", *_need]],
                on="image_id", validate="one_to_one")

# frac = the share of the ORIGINAL picture that survives the fixed 200x200 crop.
# frac = 1.0 means the source was already <=200 on its short side, so nothing was cut.
# frac near 0 means the crop is a tiny window onto a big picture (e.g. a RAISE scan).
M["frac"]   = (200 * 200) / (M.source_width * M.source_height)
M["logfrac"] = np.log10(M.frac)
M["source"] = M.image_id.str.split("/").str[0]

M[["image_id", "label", "source", "source_width", "source_height", "frac"]].head()

## Step 1 — is the shortcut even available?

If we throw away every pixel and score each image by `frac` alone (nothing but "how much of
the original photo survived the crop"), how well would that already separate real from fake?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.1))

bins = np.linspace(0, 1, 26)
axes[0].hist(M.loc[M.label == 0, "frac"], bins=bins, color=REAL, alpha=0.75, label="real")
axes[0].hist(M.loc[M.label == 1, "frac"], bins=bins, color=FAKE, alpha=0.55, label="fake")
axes[0].set_xlabel("crop fraction  (1.0 = whole image kept)")
axes[0].set_ylabel("count")
axes[0].set_title("Real photos get cropped much harder than fakes")
axes[0].legend(frameon=False, fontsize=8)

order = M.sort_values("frac")
axes[1].plot(np.linspace(0, 1, (order.label == 0).sum()),
             np.sort(order.loc[order.label == 0, "frac"]), color=REAL, label="real")
axes[1].plot(np.linspace(0, 1, (order.label == 1).sum()),
             np.sort(order.loc[order.label == 1, "frac"]), color=FAKE, label="fake")
axes[1].set_xlabel("percentile within class")
axes[1].set_ylabel("crop fraction")
axes[1].set_title("Same thing, sorted")
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

geom_auroc = auroc(M.frac, M.label)
uncropped_fake = (M.loc[M.label == 1, "frac"] >= 0.999).mean()
uncropped_real = (M.loc[M.label == 0, "frac"] >= 0.999).mean()
print(f"AUROC using crop fraction ALONE, no pixels at all: {geom_auroc:.3f}")
print(f"fakes scored at frac=1.0 (no crop happened):  {uncropped_fake:6.1%}")
print(f"reals scored at frac=1.0 (no crop happened):  {uncropped_real:6.1%}")

**Brief.** Yes, the shortcut exists: crop fraction alone gives AUROC 0.78 — better than
AEROBLADE (0.59) and close to cnndetection (0.79) on actual pixels. 44% of fakes are small
enough to survive the crop whole, versus 1.4% of real photos. (Local sample: 0.85.) So we have
to rule out that the detectors use it.

## Step 2 — do the detectors actually use it?

Real photos only. Fakes tend to be *uncropped* (large `frac`), so a detector exploiting the
shortcut would score large-`frac` images as more fake, i.e. a **positive** correlation between
`log frac` and score. A **negative** correlation means the opposite: the harder a real photo is
cropped, the *more fake* it looks — not what the shortcut predicts.

In [ ]:
reals = M[M.label == 0]
corr = {d: spearman(reals.logfrac, reals[d]) for d in DETECTORS}

print("correlation of (log crop fraction) with score, among REAL photos only:")
print("positive = shortcut-consistent (harder crop -> scored more real)")
print("negative = shortcut-backwards (harder crop -> scored MORE fake)\n")
for d, r in corr.items():
    flag = "shortcut-consistent" if r > 0.1 else ("backwards" if r < -0.1 else "~flat")
    print(f"  {d:26s} rho = {r:+.3f}   ({flag})")

fig, ax = plt.subplots(figsize=(4.3, 3.4))
d = "dmimagedetection_progan"
ax.scatter(reals.logfrac, reals[d], s=10, color=REAL, alpha=0.5)
ax.set_xlabel("log10(crop fraction)  -- more negative = harder crop")
ax.set_ylabel(f"{d} score (higher = 'more fake')")
ax.set_title("Real photos: harder-cropped ones score MORE fake, not less")
fig.tight_layout()
plt.show()

**Brief.** Almost no relationship. cnndetection (−0.07) and dmid_latent (+0.02) are flat;
univfd and dmid_progan are weakly backwards (−0.13, −0.12); only AEROBLADE leans the shortcut
way (+0.20), and modestly.

On the local sample, all four classifiers were strongly backwards (−0.36 to −0.57). That
difference comes from RAISE: there it was a third of the real photos and is both the
most-cropped and the most fake-looking source. Here RAISE is 0.6% of real photos and the
correlation mostly disappears — so the local "harder crop → looks fake" effect was a RAISE
effect, not a crop effect.

## Step 3 — the acid test: remove the shortcut and see what's left

Restrict to the band of `frac` where real and fake photos overlap a lot (so "how much was
cropped" stops being a useful guess on its own), and recompute AUROC there. If the detectors
were relying on the shortcut, taking it away should tank their AUROC. If they're reading actual
content, it should hold up — or even improve, since Step 2 showed the shortcut was working
*against* real photos.

In [ ]:
lo, hi = 0.10, 0.35
band = M[(M.frac >= lo) & (M.frac < hi)]
n0, n1 = (band.label == 0).sum(), (band.label == 1).sum()

rows = []
for d in DETECTORS:
    rows.append({"detector": d,
                 "AUROC (full sample)": auroc(M[d], M.label),
                 f"AUROC (frac in [{lo},{hi}), n={n0} real/{n1} fake)": auroc(band[d], band.label)})
tab = pd.DataFrame(rows).set_index("detector")

x = np.arange(len(DETECTORS))
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(x - 0.18, tab.iloc[:, 0], width=0.36, color="#9aa0a6", label="full sample (crop confound present)")
ax.bar(x + 0.18, tab.iloc[:, 1], width=0.36, color=FAKE, label="crop fraction matched across classes")
ax.axhline(0.5, color="black", lw=0.8, ls=":")
ax.set_xticks(x); ax.set_xticklabels([d.replace("dmimagedetection_", "dmid_") for d in DETECTORS],
                                       rotation=20, ha="right", fontsize=8)
ax.set_ylabel("AUROC")
ax.set_ylim(0.4, 1.0)
ax.set_title("Removing the crop confound does not hurt the panel -- it helps")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

tab.round(3)

**Brief.** Nothing collapses. Inside the band (129,213 real / 26,220 fake):
dmid_latent 0.89 → 0.94, dmid_progan 0.89 → 0.93, cnndetection 0.79 → 0.81, AEROBLADE
0.59 → 0.78; UnivFD is the only one to dip (0.80 → 0.78). The panel's separation is not coming
from crop geometry.

Caveat: restricting `frac` also changes *which generators* are compared (the band holds images
of roughly 340–630 px per side, e.g. 512×512 diffusion outputs). AEROBLADE's jump most likely
reflects that — it favours latent-diffusion outputs — rather than a crop effect.

## Step 4 — which real photos cause the trouble?

**RAISE**: camera-raw, very high resolution (2000×3008 to 4928×3264), no history of web JPEG
re-encoding — the closest thing in this set to archival material. Below, a threshold is set so
each detector falsely flags 5% of COCO, then applied to the other sources.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.2))
sources = ["COCO2017_train", "LAION-400M", "RAISE"]
colors = ["#3b6ea5", "#6f9bc4", "#0b2f52"]  # light -> dark, same hue family as REAL

fpr_rows = []
for d in DETECTORS:
    thr = np.quantile(M.loc[M.source == "COCO2017_train", d], 0.95)  # threshold = 5% FPR on COCO
    row = {"detector": d, "threshold (5% FPR on COCO)": thr}
    for s in sources:
        row[s] = (M.loc[M.source == s, d] > thr).mean()
    row["fakes flagged (TPR)"] = (M.loc[M.label == 1, d] > thr).mean()
    fpr_rows.append(row)
fpr_tab = pd.DataFrame(fpr_rows).set_index("detector")

x = np.arange(len(DETECTORS))
w = 0.25
for i, s in enumerate(sources):
    ax.bar(x + (i - 1) * w, fpr_tab[s], width=w, color=colors[i], label=s)
ax.set_xticks(x); ax.set_xticklabels([d.replace("dmimagedetection_", "dmid_") for d in DETECTORS],
                                       rotation=20, ha="right", fontsize=8)
ax.set_ylabel("false-positive rate")
ax.set_title("False-positive rate by authentic source, threshold fixed at 5% FPR on COCO")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

fpr_tab.round(3)

**Brief.** Same threshold, very different false-alarm rates:

- **RAISE: 20–58% falsely flagged** — univfd 58%, dmid_latent 58%, dmid_progan 49%,
  cnndetection 29%, AEROBLADE 20%.
- LAION: 8–10% on the three GAN-leaning classifiers, but 31% on dmid_latent and 46% on
  AEROBLADE.
- UnivFD flags *more* RAISE photos (58%) than actual fakes (42%).

These rates match the local sample within ~4 points (except dmid_latent, 44% → 58%). This is
the most stable result so far: the detectors read "unusual capture/processing pipeline" as a
generation artifact — the archival-domain-gap problem.

## Brief: summary in plain words

- **The shortcut exists** (crop fraction alone: AUROC 0.78), but **the detectors are not using
  it**: scores barely move with crop fraction among real photos, and removing the shortcut
  leaves every detector's AUROC about the same or higher.
- **The real problem is RAISE.** At a threshold with 5% false alarms on COCO, 20–58% of genuine
  camera-raw photos are flagged as fake. This held from 612 to 355,638 images.

### Caveats
- Correlational. It shows the panel *isn't* exploiting crop geometry here, not that no detector
  ever could.
- RAISE is the only high-resolution source and only 1,000 images, so "RAISE-ness" and "heavily
  cropped" still cannot be fully separated with this data.
- AUROC is used only as a diagnostic; Cllr/DET/Tippett are elsewhere.

### Suggested next step (~20 min, optional)
Take a handful of RAISE images, crop the *same* image at several `frac` values, and rescore.
Same content, only crop fraction changes — that turns this into a causal test.